In [1]:
import json
import torch
import datasets
import transformers
from tqdm import tqdm

In [9]:
# 加载tokenizer
tokenizer = transformers.AutoTokenizer.from_pretrained("tokenizers/merged_tokenizer")
model = transformers.AutoModelForCausalLM.from_pretrained(
    # "saved/continue_pretrain/cpt-qwen-0.1B/"
    "saved/pretrain/pretrain-qwen-0.1B/"
).to("cuda")

In [10]:
# 加载医疗测试数据
test_data = datasets.load_dataset(
    "json", data_files="data/test_encyclopedia.json"
)

In [7]:
# 通用领域数据
test_data_general = datasets.load_dataset(
    "json", data_files="data/pretrain_test_1w.json"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
# 定义参数
stride = 256
max_length = 512
total_ppls = []

for line in tqdm(test_data["train"], total=len(test_data)):
    text = line["text"]
    encodings = tokenizer(text, return_tensors='pt')
    seq_len = encodings.input_ids.size(1)

    nll_sum = 0.0
    n_tokens = 0
    prev_end_loc = 0
    for begin_loc in range(0, seq_len, stride):
        
        # 计算要预测的区间起点和终点
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to("cuda")
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        # 得到这一段tokens的loss
        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss

        num_valid_tokens = (target_ids != -100).sum().item() # 统计target里面有多少有效token
        batch_size = target_ids.size(0)
        num_loss_tokens = num_valid_tokens - batch_size  # 减去因为内部label right shift的tokens

        # 累积loss和tokens
        nll_sum += neg_log_likelihood * num_loss_tokens
        n_tokens += num_loss_tokens

        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    avg_nll = nll_sum / n_tokens # avg loss
    ppl = torch.exp(avg_nll) # 一条样本的ppl
    total_ppls.append(ppl.item())

average_ppl = sum(total_ppls) / len(total_ppls)
print(f"文档数: {len(total_ppls)}, 平均的PPL: {average_ppl}")

500it [00:09, 50.27it/s]                     

文档数: 500, 平均的PPL: 12.513849229812623


In [8]:
# 定义参数
stride = 256
max_length = 512
total_ppls = []

for line in tqdm(test_data_general["train"], total=len(test_data)):
    text = line["text"]
    encodings = tokenizer(text, return_tensors='pt')
    seq_len = encodings.input_ids.size(1)

    nll_sum = 0.0
    n_tokens = 0
    prev_end_loc = 0
    for begin_loc in range(0, seq_len, stride):
        
        # 计算要预测的区间起点和终点
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to("cuda")
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        # 得到这一段tokens的loss
        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss

        num_valid_tokens = (target_ids != -100).sum().item() # 统计target里面有多少有效token
        batch_size = target_ids.size(0)
        num_loss_tokens = num_valid_tokens - batch_size  # 减去因为内部label right shift的tokens

        # 累积loss和tokens
        nll_sum += neg_log_likelihood * num_loss_tokens
        n_tokens += num_loss_tokens

        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    avg_nll = nll_sum / n_tokens # avg loss
    ppl = torch.exp(avg_nll) # 一条样本的ppl
    total_ppls.append(ppl.item())

average_ppl = sum(total_ppls) / len(total_ppls)
print(f"文档数: {len(total_ppls)}, 平均的PPL: {average_ppl}")

11496it [01:54, 100.17it/s]          

文档数: 11496, 平均的PPL: 19.946458388915804


In [11]:
# 定义参数
stride = 256
max_length = 512
total_ppls = []

for line in tqdm(test_data["train"], total=len(test_data)):
    text = line["text"]
    encodings = tokenizer(text, return_tensors='pt')
    seq_len = encodings.input_ids.size(1)

    nll_sum = 0.0
    n_tokens = 0
    prev_end_loc = 0
    for begin_loc in range(0, seq_len, stride):
        
        # 计算要预测的区间起点和终点
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to("cuda")
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        # 得到这一段tokens的loss
        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss

        num_valid_tokens = (target_ids != -100).sum().item() # 统计target里面有多少有效token
        batch_size = target_ids.size(0)
        num_loss_tokens = num_valid_tokens - batch_size  # 减去因为内部label right shift的tokens

        # 累积loss和tokens
        nll_sum += neg_log_likelihood * num_loss_tokens
        n_tokens += num_loss_tokens

        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    avg_nll = nll_sum / n_tokens # avg loss
    ppl = torch.exp(avg_nll) # 一条样本的ppl
    total_ppls.append(ppl.item())

average_ppl = sum(total_ppls) / len(total_ppls)
print(f"文档数: {len(total_ppls)}, 平均的PPL: {average_ppl}")

500it [00:09, 50.43it/s]             

文档数: 500, 平均的PPL: 89.99840065383911
